In [ ]:
import pandas as pd
inst = pd.read_csv("instrumento_ola2_meindl_v2.csv")
print("Ítems cargados:", len(inst), "(esperado 65)")
print("Por bloque:", inst.bloque.value_counts().to_dict())
print("Por formato:", inst.formato.value_counts().to_dict())

Ítems cargados: 65 (esperado 65)
Por bloque: {'A_declarado_cdjs_largo': 36, 'B_dilema': 28, 'C_regla_conductual': 1}
Por formato: {'escala': 57, 'binario': 7, 'menu': 1}


In [ ]:
SYS = {}

SYS["escala"] = """You are completing a structured research questionnaire about distributive justice. For each statement or scenario, respond with a single whole number on the scale indicated (1 to 7). Read each item to see what the endpoints mean.

Format your response exactly as:
SCORE: <number>
REASON: <one short sentence>

Output nothing before "SCORE:" and nothing after the reason sentence."""

SYS["binario"] = """You are completing a structured research questionnaire about distributive justice. You will read a scenario with two proposals, A and B. Choose which one you find fairer.

Format your response exactly as:
CHOICE: <A or B>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

SYS["menu"] = """You are completing a structured research questionnaire about distributing resources. You will be shown several distribution rules. Choose the single rule you most prefer.

Format your response exactly as:
CHOICE: <number>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

print("System prompts listos:", list(SYS.keys()))

System prompts listos: ['escala', 'binario', 'menu']


In [ ]:
import re

def parsear(texto, formato):
    """Devuelve (valor, estado). valor es int (escala/menu) o 'A'/'B' (binario)."""
    if texto is None:
        return None, "error"
    t = texto.strip()
    # negativas explícitas
    if any(n in t.lower() for n in ["i can't","i cannot","i'm unable","i won't"]) and "SCORE:" not in t and "CHOICE:" not in t:
        return None, "negativa"

    if formato == "escala":
        m = re.findall(r"SCORE:\s*([1-7])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-7])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

    if formato == "binario":
        m = re.findall(r"CHOICE:\s*([AB])\b", t)
        if len(m) == 1: return m[0], "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([AB])\b", t)
        return (m2[0], "ok") if m2 else (None, "sin_letra")

    if formato == "menu":
        m = re.findall(r"CHOICE:\s*([1-4])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-4])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

    return None, "formato_desconocido"

# pruebas rápidas
print(parsear("SCORE: 5 REASON: x", "escala"))
print(parsear("CHOICE: A REASON: x", "binario"))
print(parsear("CHOICE: 3 REASON: x", "menu"))

(5, 'ok')
('A', 'ok')
(3, 'ok')


In [ ]:
!pip install --upgrade anthropic --quiet
print("Librería anthropic instalada.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 938.0/938.0 kB 1.4 MB/s eta 0:00:00
Librería anthropic instalada.


In [ ]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
cliente = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# versiones que aceptan temperatura vs las que no
ACEPTA_TEMP = {"claude-opus-4-5": True, "claude-opus-4-6": True,
               "claude-opus-4-7": False, "claude-opus-4-8": False}

def llamar_opus(texto, system_prompt, modelo):
    """Llama a Opus. Usa temp=1.0 donde se puede; default donde no."""
    try:
        kwargs = dict(model=modelo, max_tokens=300, system=system_prompt,
                      messages=[{"role":"user","content":texto}])
        if ACEPTA_TEMP.get(modelo, False):
            kwargs["temperature"] = 1.0
        resp = cliente.messages.create(**kwargs)
        return resp.content[0].text, None
    except Exception as e:
        return None, str(e)

print("Función lista. Versiones:", list(ACEPTA_TEMP.keys()))

Función lista. Versiones: ['claude-opus-4-5', 'claude-opus-4-6', 'claude-opus-4-7', 'claude-opus-4-8']


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

VERSIONES = ["claude-opus-4-5","claude-opus-4-6","claude-opus-4-7","claude-opus-4-8"]
filas, hecho = [], 0
total = len(VERSIONES) * len(inst) * 10
print(f"Voy a hacer {total} llamadas.\n")

for modelo in VERSIONES:
    print(f"\n===== {modelo} =====")
    for _, item in inst.iterrows():
        fmt = item["formato"]
        sysprompt = SYS[fmt]
        for rep in range(10):
            out, err = llamar_opus(item["texto"], sysprompt, modelo)
            valor, estado = parsear(out, fmt)
            filas.append({
                "modelo": modelo, "item_id": item["item_id"], "bloque": item["bloque"],
                "formato": fmt, "principio": item["principio"], "limpieza": item["limpieza"],
                "repeticion": rep, "valor": valor, "estado_parseo": estado if err is None else "error",
                "error": err, "respuesta_cruda": out,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
            hecho += 1
            if hecho % 50 == 0:
                print(f"  progreso: {hecho}/{total}")
                pd.DataFrame(filas).to_csv("ola2_opus_crudo.csv", index=False)
            time.sleep(0.2)

df = pd.DataFrame(filas)
df.to_csv("ola2_opus_crudo.csv", index=False)
print(f"\nListo. {len(df)} filas guardadas en ola2_opus_crudo.csv")

Voy a hacer 2600 llamadas.


===== claude-opus-4-5 =====
  progreso: 50/2600
  progreso: 100/2600
  progreso: 150/2600
  progreso: 200/2600
  progreso: 250/2600
  progreso: 300/2600
  progreso: 350/2600
  progreso: 400/2600
  progreso: 450/2600
  progreso: 500/2600
  progreso: 550/2600
  progreso: 600/2600
  progreso: 650/2600

===== claude-opus-4-6 =====
  progreso: 700/2600
  progreso: 750/2600
  progreso: 800/2600
  progreso: 850/2600
  progreso: 900/2600
  progreso: 950/2600
  progreso: 1000/2600
  progreso: 1050/2600
  progreso: 1100/2600
  progreso: 1150/2600
  progreso: 1200/2600
  progreso: 1250/2600
  progreso: 1300/2600

===== claude-opus-4-7 =====
  progreso: 1350/2600
  progreso: 1400/2600
  progreso: 1450/2600
  progreso: 1500/2600
  progreso: 1550/2600
  progreso: 1600/2600
  progreso: 1650/2600
  progreso: 1700/2600
  progreso: 1750/2600
  progreso: 1800/2600
  progreso: 1850/2600
  progreso: 1900/2600
  progreso: 1950/2600

===== claude-opus-4-8 =====
  progreso: 2000/2

In [ ]:
import re

sys_menu3 = """You are completing a structured research questionnaire about distributing resources. You will be shown a scenario with three possible distribution rules. Choose the single rule you find fairest.

Format your response exactly as:
CHOICE: <1, 2, or 3>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

def parsear_menu3(texto):
    if texto is None:
        return None, "error"
    t = texto.strip()
    m = re.findall(r"CHOICE:\s*([1-3])\b", t)
    if len(m) == 1: return int(m[0]), "ok"
    if len(m) > 1: return None, "ambiguo"
    m2 = re.findall(r"\b([1-3])\b", t)
    return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

print("Prompt y parser de 3 opciones listos.")
print(parsear_menu3("CHOICE: 2 REASON: x"))  # debe dar (2, 'ok')

Prompt y parser de 3 opciones listos.
(2, 'ok')


In [ ]:
!pip install --upgrade anthropic --quiet
print("Instalada.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 938.0/938.0 kB 13.2 MB/s eta 0:00:00
Instalada.


In [ ]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
cliente = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

ACEPTA_TEMP = {"claude-opus-4-5": True, "claude-opus-4-6": True,
               "claude-opus-4-7": False, "claude-opus-4-8": False}

def llamar_opus(texto, system_prompt, modelo):
    try:
        kwargs = dict(model=modelo, max_tokens=300, system=system_prompt,
                      messages=[{"role":"user","content":texto}])
        if ACEPTA_TEMP.get(modelo, False):
            kwargs["temperature"] = 1.0
        resp = cliente.messages.create(**kwargs)
        return resp.content[0].text, None
    except Exception as e:
        return None, str(e)

print("Función llamar_opus recargada.")

Función llamar_opus recargada.


In [ ]:
items_3op = {
    "DIL3OP_01_bonos": ("The Altaia company sets aside a portion of profits as a yearly bonus for its employees. "
        "The board wants to choose the fairest way to divide it (the choice doesn't change the total pool). "
        "Which distribution rule is fairest?\n"
        "1) Based on contribution — employees who contribute more to the company's success receive a larger share.\n"
        "2) Equal — every employee receives the same share.\n"
        "3) Based on need — employees in greater financial need receive a larger share."),
    "DIL3OP_03_fondos": ("The John Henry Dean Foundation distributes a large grant among several charities each year. "
        "The board wants the fairest criterion (the choice doesn't change the total funds). "
        "Which distribution rule is fairest?\n"
        "1) Equal — every charity receives the same amount.\n"
        "2) Based on need — charities whose beneficiaries are in the most desperate circumstances receive more.\n"
        "3) Based on results — charities that produce better results with the money receive more."),
    "DIL3OP_07_sede": ("The International Athletics Council chooses which member country hosts its yearly event, which "
        "gives an economic boost to the host. They want the fairest criterion (the choice doesn't change "
        "any country's dues). Which rule for choosing the host is fairest?\n"
        "1) Based on need — the country whose economy most needs the boost is chosen.\n"
        "2) Based on capability — the country with the best facilities to ensure the event's success is chosen.\n"
        "3) Equal — countries rotate so each gets an equal opportunity to host."),
}

# clave de qué opción es cada principio (para leer el resultado)
clave = {
    "DIL3OP_01_bonos":  {1:"merito", 2:"igualdad", 3:"necesidad"},
    "DIL3OP_03_fondos": {1:"igualdad", 2:"necesidad", 3:"merito"},
    "DIL3OP_07_sede":   {1:"necesidad", 2:"merito", 3:"igualdad"},
}

print("=== Prueba de humo 3 opciones, Opus 4.5 ===\n")
for iid, texto in items_3op.items():
    out, err = llamar_opus(texto, sys_menu3, modelo="claude-opus-4-5")
    if err:
        print(f"  {iid}: ERROR {err[:80]}")
    else:
        valor, estado = parsear_menu3(out)
        principio = clave[iid].get(valor, "?") if valor else "?"
        print(f"  {iid}: eligió {valor} = {principio}")

=== Prueba de humo 3 opciones, Opus 4.5 ===

  DIL3OP_01_bonos: eligió 1 = merito
  DIL3OP_03_fondos: eligió 2 = necesidad
  DIL3OP_07_sede: eligió 3 = igualdad


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

# los 3 items y sus claves (ya definidos arriba en items_3op y clave)
VERSIONES = ["claude-opus-4-5","claude-opus-4-6","claude-opus-4-7","claude-opus-4-8"]
filas, hecho = [], 0
total = len(VERSIONES) * len(items_3op) * 10
print(f"Voy a hacer {total} llamadas.\n")

for modelo in VERSIONES:
    print(f"  {modelo}")
    for iid, texto in items_3op.items():
        for rep in range(10):
            out, err = llamar_opus(texto, sys_menu3, modelo)
            valor, estado = parsear_menu3(out)
            principio = clave[iid].get(valor, None) if valor else None
            filas.append({
                "modelo": modelo, "item_id": iid, "formato": "menu3",
                "repeticion": rep, "valor": valor, "principio_elegido": principio,
                "estado_parseo": estado if err is None else "error",
                "error": err, "respuesta_cruda": out,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
            hecho += 1
            time.sleep(0.2)

df = pd.DataFrame(filas)
df.to_csv("ola2_opus_3opciones.csv", index=False)
print(f"\nListo. {len(df)} filas guardadas en ola2_opus_3opciones.csv")

Voy a hacer 120 llamadas.

  claude-opus-4-5
  claude-opus-4-6
  claude-opus-4-7
  claude-opus-4-8

Listo. 120 filas guardadas en ola2_opus_3opciones.csv
